# Microphone pitch classification and spectrogram reconstruction

In [ ]:
# Colab setup. Run after uploading/cloning the full project folder.
# !pip install -r requirements.txt

from pathlib import Path
import sys

def find_project_root(start=Path.cwd()):
    start = Path(start).resolve()
    for path in (start, *start.parents):
        if (path / "nsynth_pipeline").is_dir():
            return path
    raise FileNotFoundError("Could not find nsynth_pipeline. Upload/clone the full project, then run from inside it.")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
import base64
import io

import matplotlib.pyplot as plt
import soundfile as sf
import torch
from google.colab import output
from IPython.display import Audio, display

from nsynth_pipeline.convexnet import freeze_module, load_convnext_backbone
from nsynth_pipeline.data import LogMelTransform, MelConfig, NOTE_NAMES
from nsynth_pipeline.models import ConditionalSpectrogramVAE, ConvNeXtNoteClassifier

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
if device.type == "cuda":
    print(torch.cuda.get_device_name(0))

In [ ]:
CLASSIFIER_CHECKPOINT = "checkpoints/convnext_pitch_classifier.pt"
VAE_CHECKPOINT = "checkpoints/conditional_spectrogram_vae.pt"
RECORD_SECONDS = 4

mel_config = MelConfig(clip_seconds=RECORD_SECONDS)
log_mel = LogMelTransform(mel_config)

In [ ]:
backbone = load_convnext_backbone()
freeze_module(backbone)
classifier = ConvNeXtNoteClassifier(backbone).to(device)
classifier_checkpoint = torch.load(CLASSIFIER_CHECKPOINT, map_location=device)
classifier.load_state_dict(classifier_checkpoint["model_state_dict"])
classifier.eval()

vae_checkpoint = torch.load(VAE_CHECKPOINT, map_location=device)
vae = ConditionalSpectrogramVAE(
    input_shape=tuple(vae_checkpoint["input_shape"]),
    latent_dim=int(vae_checkpoint.get("latent_dim", 128)),
).to(device)
vae.load_state_dict(vae_checkpoint["model_state_dict"])
vae.eval()

In [ ]:
RECORD_WAV_JS = """
async function recordWav(seconds) {
  const stream = await navigator.mediaDevices.getUserMedia({audio: true});
  const context = new AudioContext();
  const source = context.createMediaStreamSource(stream);
  const processor = context.createScriptProcessor(4096, 1, 1);
  const chunks = [];

  processor.onaudioprocess = event => {
    chunks.push(new Float32Array(event.inputBuffer.getChannelData(0)));
  };

  source.connect(processor);
  processor.connect(context.destination);
  await new Promise(resolve => setTimeout(resolve, seconds * 1000));

  processor.disconnect();
  source.disconnect();
  stream.getTracks().forEach(track => track.stop());

  const length = chunks.reduce((sum, chunk) => sum + chunk.length, 0);
  const samples = new Float32Array(length);
  let offset = 0;
  for (const chunk of chunks) {
    samples.set(chunk, offset);
    offset += chunk.length;
  }

  const wav = encodeWav(samples, context.sampleRate);
  context.close();
  let binary = '';
  const bytes = new Uint8Array(wav);
  for (let i = 0; i < bytes.byteLength; i++) binary += String.fromCharCode(bytes[i]);
  return btoa(binary);
}

function encodeWav(samples, sampleRate) {
  const buffer = new ArrayBuffer(44 + samples.length * 2);
  const view = new DataView(buffer);

  writeString(view, 0, 'RIFF');
  view.setUint32(4, 36 + samples.length * 2, true);
  writeString(view, 8, 'WAVE');
  writeString(view, 12, 'fmt ');
  view.setUint32(16, 16, true);
  view.setUint16(20, 1, true);
  view.setUint16(22, 1, true);
  view.setUint32(24, sampleRate, true);
  view.setUint32(28, sampleRate * 2, true);
  view.setUint16(32, 2, true);
  view.setUint16(34, 16, true);
  writeString(view, 36, 'data');
  view.setUint32(40, samples.length * 2, true);

  let offset = 44;
  for (let i = 0; i < samples.length; i++, offset += 2) {
    const sample = Math.max(-1, Math.min(1, samples[i]));
    view.setInt16(offset, sample < 0 ? sample * 0x8000 : sample * 0x7fff, true);
  }
  return buffer;
}

function writeString(view, offset, string) {
  for (let i = 0; i < string.length; i++) view.setUint8(offset + i, string.charCodeAt(i));
}
"""


def record_wav(seconds):
    wav_base64 = output.eval_js(RECORD_WAV_JS + f"recordWav({seconds})")
    wav_bytes = base64.b64decode(wav_base64)
    audio, sample_rate = sf.read(io.BytesIO(wav_bytes), dtype="float32")
    return torch.from_numpy(audio), sample_rate, wav_bytes

In [ ]:
waveform, sample_rate, wav_bytes = record_wav(RECORD_SECONDS)
display(Audio(wav_bytes, rate=sample_rate))

spectrogram = log_mel(waveform, sample_rate).unsqueeze(0).to(device)
with torch.no_grad():
    outputs = classifier(spectrogram)
    predicted_note = outputs["note_logits"].argmax(dim=-1)
    predicted_pitch = outputs["pitch_logits"].argmax(dim=-1)
    reconstruction = vae.generate(predicted_pitch, device=device).cpu()

print({
    "predicted_note_class": int(predicted_note.item()),
    "predicted_note": NOTE_NAMES[int(predicted_note.item())],
    "predicted_pitch": int(predicted_pitch.item()),
})

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].imshow(spectrogram[0, 0].detach().cpu(), aspect="auto", origin="lower")
axes[0].set_title("Microphone log-mel")
axes[1].imshow(reconstruction[0, 0], aspect="auto", origin="lower")
axes[1].set_title("Random-z reconstruction")
for axis in axes:
    axis.axis("off")
plt.tight_layout()